In [11]:
# Please add the necessary parameter for the model to run

# Please make sure all the modules and libraries are imported correctly


advisor_inputs = {
    "client_name": "Rahul Sharma",     
    "current_age": 25,
    "annual_premium": 150000,
    "payout_pct": 0.40,
    "life_cover_multiple": 7,
    "ppt": 12,
    "policy_term": 40,
    "expected_return": 0.18, 
    "monthly_swp": 350000,      # Target monthly SWP payout amount
    "swp_start_age": 50          # Target retirement milestone age
}

# Define explicit currency step-ups here (Format -> Policy_Year: Absolute_Increase)
# Example -> 2: 500 means in year 2, add ₹500/month on top of the base SIP
custom_schedule = {
    # 2: 500,
    # 5: 1000
}
# ═════════════════════════════════════════════════════════════════════════════════════════
# ⚙️ CORE ENGINE CODE (Safe to leave untouched below this line)
# ═════════════════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd

class GranularWealthEngine:
    def __init__(self, inputs: dict, step_up_schedule: dict):
        self.client_name = inputs.get("client_name", "Valued Client")
        self.current_age = inputs.get("current_age", 35)
        self.annual_premium = inputs.get("annual_premium", 150000)
        self.payout_pct = inputs.get("payout_pct", 0.40)
        self.life_cover_multiple = inputs.get("life_cover_multiple", 7)
        self.ppt = inputs.get("ppt", 12)
        self.policy_term = inputs.get("policy_term", 40)
        self.expected_return = inputs.get("expected_return", 0.18)
        self.monthly_swp_target = inputs.get("monthly_swp", 100000)
        self.swp_start_age = inputs.get("swp_start_age", 60)
        self.step_up_schedule = step_up_schedule
        
        self.life_cover = self.annual_premium * self.life_cover_multiple
        self.annual_payout = self.annual_premium * self.payout_pct
        self.base_monthly_sip = round(self.annual_payout / 12, 2)
        self.monthly_rate = self.expected_return / 12
        
    def run_projection(self, alternative_swp_age: int = None, custom_swp_target: float = None) -> pd.DataFrame:
        target_swp_age = alternative_swp_age if alternative_swp_age is not None else self.swp_start_age
        active_swp_payout = custom_swp_target if custom_swp_target is not None else self.monthly_swp_target
        
        records = []
        current_corpus = 0.0
        cumulative_step_up = 0.0
        
        for year in range(1, self.policy_term + 1):
            age = self.current_age + (year - 1)
            premium_paid = self.annual_premium if year <= self.ppt else 0.0
            insurance_payout = self.annual_payout
            
            year_specific_increment = self.step_up_schedule.get(year, 0.0)
            cumulative_step_up += year_specific_increment
            
            total_monthly_sip = self.base_monthly_sip + cumulative_step_up
            annual_sip_contribution = total_monthly_sip * 12
            monthly_swp = active_swp_payout if age >= target_swp_age else 0.0
            
            for month in range(1, 13):
                if current_corpus <= 0: current_corpus = 0.0
                current_corpus += total_monthly_sip
                current_corpus *= (1 + self.monthly_rate)
                
                if current_corpus >= monthly_swp: current_corpus -= monthly_swp
                else: current_corpus = 0.0
            
            net_corpus = round(current_corpus, 2)
            annual_swp_withdrawn = (active_swp_payout * 12) if age >= target_swp_age else 0.0
            status = "Corpus Exhausted" if net_corpus <= 0 else "Sustainable"
                
            records.append({
                "Policy Year": year, "Age": age, "Premium Paid (₹)": premium_paid,
                "Insurance Payout (₹)": insurance_payout, "Base Monthly SIP (₹)": self.base_monthly_sip,
                "New Step-Up Added (₹)": year_specific_increment, "Total Monthly SIP (₹)": total_monthly_sip,
                "Annual SIP Contribution (₹)": annual_sip_contribution, "Annual SWP Withdrawal (₹)": annual_swp_withdrawn,
                "Net End-of-Year Corpus (₹)": net_corpus, "Sustainability Flag": status
            })
        return pd.DataFrame(records)

    def generate_executive_summary(self, df: pd.DataFrame, max_monthly_swp: float) -> dict:
        retirement_row = df[df["Age"] == self.swp_start_age]
        corpus_at_retirement = retirement_row["Net End-of-Year Corpus (₹)"].values[0] if not retirement_row.empty else 0.0
        final_corpus = df["Net End-of-Year Corpus (₹)"].iloc[-1]
        
        r_monthly = self.monthly_rate
        remaining_years = self.policy_term - (self.swp_start_age - self.current_age)
        n_months = max(0, remaining_years * 12)
        required_corpus = self.monthly_swp_target * ((1 - (1 + r_monthly)**-n_months) / r_monthly) if n_months > 0 else 0.0
            
        exhausted_rows = df[df["Sustainability Flag"] == "Corpus Exhausted"]
        survival_status = "Corpus Exhausted" if not exhausted_rows.empty else "Sustainable"
        survives_until_year = exhausted_rows["Policy Year"].values[0] if not exhausted_rows.empty else self.policy_term
        survives_until_age = exhausted_rows["Age"].values[0] if not exhausted_rows.empty else self.current_age + self.policy_term
            
        if survival_status == "Sustainable":
            max_annual_equivalent = max_monthly_swp * 12
            formatted_strategic_note = (
                f"If you follow this investment track, you can safely withdraw a maximum of up to "
                f"₹{max_monthly_swp:,.2f}/month (₹{max_annual_equivalent:,.2f}/year) starting from age {self.swp_start_age} "
                f"without ever touching or exhausting your core wealth corpus."
            )
        else:
            if r_monthly > 0 and n_months > 0:
                factor = (1 - (1 + r_monthly) ** -n_months) / r_monthly
                exact_target_needed = (self.monthly_swp_target * factor) / (1 + r_monthly)
            else:
                exact_target_needed = 0.0
                
            shortfall_gap = max(0.0, exact_target_needed - corpus_at_retirement)
            total_months_to_retirement = max(1, (self.swp_start_age - self.current_age) * 12)
            annuity_factor = (((1 + r_monthly) ** total_months_to_retirement - 1) / r_monthly) * (1 + r_monthly)
            suggested_topup = round(shortfall_gap / annuity_factor, 2) if annuity_factor > 0 else 0.0

            formatted_strategic_note = (
                f"CRITICAL ACTION REQUIRED: This plan is currently unsustainable. To meet your target milestone, "
                f"you need to add an additional top-up investment of approximately ₹{suggested_topup:,.2f}/month "
                f"on top of the standard payouts from the policy to transition this framework into a fully sustainable model."
            )
            
        return {
            "Life Cover Amount": self.life_cover, "Monthly Insurance Payout": self.base_monthly_sip,
            "Corpus at SWP Start": corpus_at_retirement, "Target Required Corpus": round(required_corpus, 2),
            "Surplus / Shortfall": round(corpus_at_retirement - required_corpus, 2), "Final Year 40 Corpus": final_corpus,
            "Sustainability Status": survival_status, "Years Corpus Survives": survives_until_year,
            "Age Corpus Exhausted": survives_until_age, "Client Presentation Note": formatted_strategic_note
        }

    def predict_sustainability_gap(self, df: pd.DataFrame) -> dict:
        retirement_year_row = df[df["Age"] == self.swp_start_age]
        actual_corpus_at_start = 0.0
        if not retirement_year_row.empty:
            target_policy_year = retirement_year_row["Policy Year"].values[0]
            prior_row = df[df["Policy Year"] == (target_policy_year - 1)]
            actual_corpus_at_start = prior_row["Net End-of-Year Corpus (₹)"].values[0] if not prior_row.empty else 0.0

        remaining_years = self.policy_term - (self.swp_start_age - self.current_age)
        total_withdrawal_months = max(0, remaining_years * 12)
        
        r = self.monthly_rate
        exact_target_needed = (self.monthly_swp_target * ((1 - (1 + r) ** -total_withdrawal_months) / r)) / (1 + r) if r > 0 and total_withdrawal_months > 0 else 0.0
        shortfall = round(max(0.0, exact_target_needed - actual_corpus_at_start), 2)
        return {"Target Needed": round(exact_target_needed, 2), "Shortfall": shortfall}

    def calculate_inflection_point(self, df: pd.DataFrame) -> tuple:
        gap_info = self.predict_sustainability_gap(df)
        target_needed = gap_info["Target Needed"]
        for _, row in df.iterrows():
            if row["Age"] < self.swp_start_age:
                months_to_retirement = (self.swp_start_age - row["Age"]) * 12
                discounted_safety_target = target_needed / ((1 + self.monthly_rate) ** months_to_retirement)
                if row["Net End-of-Year Corpus (₹)"] < discounted_safety_target:
                    return int(row["Policy Year"]), int(row["Age"])
        return 1, self.current_age

    def find_max_monthly_swp_for_zero_residual(self) -> float:
        low_bound, high_bound, tolerance = 0.0, 10000000.0, 0.01
        best_payout = 0.0
        for _ in range(100):
            mid_payout = (low_bound + high_bound) / 2.0
            test_df = self.run_projection(custom_swp_target=mid_payout)
            if (test_df["Sustainability Flag"] == "Corpus Exhausted").iloc[:-1].any() or (test_df["Net End-of-Year Corpus (₹)"].iloc[-1] <= 0 and test_df["Net End-of-Year Corpus (₹)"].iloc[-1] == 0.0):
                high_bound = mid_payout
            else:
                best_payout = mid_payout
                low_bound = mid_payout
            if abs(high_bound - low_bound) < tolerance: break
        return round(best_payout, 2)

class DeficitBridgeEngine:
    def __init__(self, target_gap: float, start_age: int, target_age: int, expected_return: float):
        self.total_months = max(0, (target_age - start_age) * 12)
        r = expected_return / 12
        if self.total_months > 0 and r > 0:
            annuity_factor = (((1 + r) ** self.total_months - 1) / r) * (1 + r)
            self.required_monthly_investment = round(target_gap / annuity_factor, 2)
        else:
            self.required_monthly_investment = 0.0

# ── AUTO-RESOLVE RUNTIME OBJECT GENERATION FOR COMPATIBILITY ──
engine = GranularWealthEngine(advisor_inputs, custom_schedule)
max_sustainable_monthly_swp = engine.find_max_monthly_swp_for_zero_residual()
projection_matrix = engine.run_projection()
summary = engine.generate_executive_summary(projection_matrix, max_sustainable_monthly_swp)

is_unsustainable = (summary["Sustainability Status"] == "Corpus Exhausted")
top_up_start_year, top_up_start_age = engine.calculate_inflection_point(projection_matrix)
gap_analysis = engine.predict_sustainability_gap(projection_matrix)
GAP_AMOUNT = gap_analysis["Shortfall"]
bridge_calculator = DeficitBridgeEngine(GAP_AMOUNT, top_up_start_age, engine.swp_start_age, engine.expected_return)

total_bridge_months = max(1, (engine.swp_start_age - top_up_start_age) * 12)
mom_records = []
running_bridge_corpus = 0.0
for m in range(1, total_bridge_months + 1):
    total_flow = engine.base_monthly_sip + bridge_calculator.required_monthly_investment
    running_bridge_corpus += total_flow
    running_bridge_corpus *= (1 + engine.monthly_rate)
    annual_marker = round(running_bridge_corpus, 2) if m % 12 == 0 else ""
    mom_records.append({
        "Total Month": m, "Age Metric": top_up_start_age + (m // 12), "Base Monthly SIP (₹)": engine.base_monthly_sip,
        "Top-Up Required (₹)": bridge_calculator.required_monthly_investment, "Total Monthly Combined (₹)": total_flow,
        "Closing Balance (₹)": round(running_bridge_corpus, 2), "Annual Net Corpus Mapping (₹)": annual_marker
    })
mom_matrix = pd.DataFrame(mom_records)

In [12]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def format_to_lakhs(value, pos=None):
    abs_val = abs(value)
    if abs_val >= 100000:
        lakh_val = value / 100000
        return f"₹{int(lakh_val)}L" if round(lakh_val, 1).is_integer() else f"₹{lakh_val:.1f}L"
    return f"₹{value:,.0f}" if abs_val > 0 else "₹0"

# ── CHART 1: PRIMARY PORTFOLIO TRAJECTORY ──
fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(projection_matrix["Age"], projection_matrix["Net End-of-Year Corpus (₹)"], color="#1F497D", linewidth=2.5, label="Projected Wealth Corpus")
ax1.axvline(x=advisor_inputs["swp_start_age"], color="#C00000", linestyle="--", alpha=0.7, label=f"SWP Start (Age {advisor_inputs['swp_start_age']})")
ax1.set_title(f"40-Year Financial Trajectory Summary: {advisor_inputs['client_name']}", fontsize=12, fontweight='bold', pad=15)
ax1.set_xlabel("Client Age (Years)", fontsize=10)
ax1.set_ylabel("Net Portfolio Value (₹)", fontsize=10)
ax1.grid(True, linestyle=":", alpha=0.6, color="#DDDDDD")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(format_to_lakhs))

milestone_ages = set([projection_matrix["Age"].iloc[0], advisor_inputs["swp_start_age"], projection_matrix["Age"].iloc[-1]]).union(set(projection_matrix["Age"].iloc[4::5]))
for idx, row in projection_matrix.iterrows():
    c_age = int(row["Age"])
    c_corpus = row["Net End-of-Year Corpus (₹)"]
    if c_age in milestone_ages:
        offset = 12 if c_age < advisor_inputs["swp_start_age"] else -18
        ax1.annotate(format_to_lakhs(c_corpus), xy=(c_age, c_corpus), xytext=(0, offset), textcoords="offset points",
                    ha="center", va="center", fontsize=8, fontweight="bold", color="#222222",
                    bbox=dict(boxstyle="round,pad=0.2", fc="#FFFFFF", ec="#CCCCCC", lw=0.5, alpha=0.85))
ax1.legend(loc="upper left", frameon=True, fontsize=9)
plt.tight_layout()
fig1.savefig("chart1.png", dpi=120)

# ── CHART 2: SUPPLEMENTARY BRIDGING ACCOUNT (GATED) ──
if is_unsustainable and not mom_matrix.empty:
    fig2, ax2 = plt.subplots(figsize=(10, 5))
    ax2.plot(mom_matrix["Total Month"], mom_matrix["Closing Balance (₹)"], color="#006100", linewidth=2, label="Deficit Bridge Growth")
    ax2.set_title("Month-on-Month Deficit Bridge Accumulation Trajectory", fontsize=12, fontweight='bold', pad=15)
    ax2.set_xlabel("Timeline (Total Months Passed)", fontsize=10)
    ax2.set_ylabel("Bridge Value (₹)", fontsize=10)
    ax2.grid(True, linestyle=":", alpha=0.6, color="#DDDDDD")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(format_to_lakhs))
    
    t_months = mom_matrix["Total Month"].iloc[-1]
    for m_val in [1, int(t_months / 2), t_months]:
        m_row = mom_matrix[mom_matrix["Total Month"] == m_val]
        if not m_row.empty:
            c_bal = m_row["Closing Balance (₹)"].values[0]
            ax2.annotate(format_to_lakhs(c_bal), xy=(m_val, c_bal), xytext=(0, 12), textcoords="offset points", ha="center",
                        fontsize=8, fontweight="bold", color="#006100", bbox=dict(boxstyle="round,pad=0.2", fc="#F0FDF4", ec="#BBF7D0", lw=0.5, alpha=0.9))
    ax2.legend(loc="upper left", frameon=True, fontsize=9)
    plt.tight_layout()
    fig2.savefig("chart2.png", dpi=120)
plt.close('all')

In [13]:
import os
import openpyxl
from openpyxl.drawing.image import Image
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

excel_filename = f"{advisor_inputs['client_name'].replace(' ', '_')}_Wealth_Plan.xlsx"
wb = openpyxl.Workbook()

ws_summary = wb.active
ws_summary.title = "Executive Summary"
ws_projection = wb.create_sheet(title="40-Year Annual Projection")
ws_graphs = wb.create_sheet(title="Graphs")
ws_mom = wb.create_sheet(title="MoM Deficit Bridge Ledger") if is_unsustainable else None

for ws in [ws_summary, ws_projection, ws_graphs] + ([ws_mom] if ws_mom else []):
    ws.views.sheetView[0].showGridLines = True

# Formatting Tokens
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
title_font = Font(name="Calibri", size=15, bold=True, color="1F497D")
sub_title_font = Font(name="Calibri", size=11, italic=True, color="555555")
regular_font = Font(name="Calibri", size=11)
center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
right_align = Alignment(horizontal="right", vertical="center")
thin_border = Border(left=Side(style='thin', color='DDDDDD'), right=Side(style='thin', color='DDDDDD'), top=Side(style='thin', color='DDDDDD'), bottom=Side(style='thin', color='DDDDDD'))

# Populate Sheet 1 (Summary Block)
ws_summary.cell(row=2, column=2, value="WEALTH PLANNER EXECUTIVE DASHBOARD").font = title_font
ws_summary.cell(row=3, column=2, value=f"Strategic Advisory Matrix for {advisor_inputs['client_name']}").font = sub_title_font

summary_kpis = [
    ("Client Name", advisor_inputs["client_name"], "Text"),
    ("Current Age", advisor_inputs["current_age"], "Integer"),
    ("Expected Portfolio Growth Rate", advisor_inputs["expected_return"], "Percentage"),
    ("Retirement SWP Start Age", advisor_inputs["swp_start_age"], "Integer"),
    ("Initial Monthly Base Payout", summary["Monthly Insurance Payout"], "Currency"),
    ("Total Life Cover Provided", summary["Life Cover Amount"], "Currency"),
    (f"Capital Balance at Age {advisor_inputs['swp_start_age']}", summary["Corpus at SWP Start"], "Currency"),
    ("Required Balance Target for SWP", summary["Target Required Corpus"], "Currency"),
    ("Calculated Shortfall / Surplus", summary["Surplus / Shortfall"], "Currency"),
    ("Strategy Sustainability Status", summary["Sustainability Status"].upper(), "Alert" if is_unsustainable else "StatusText"),
]
if is_unsustainable:
    summary_kpis.extend([("MoM Supplementary Fix", bridge_calculator.required_monthly_investment, "Currency"), ("Target Gap Bridged", GAP_AMOUNT, "Currency")])
else:
    summary_kpis.append(("Max Safe Monthly Capacity", max_sustainable_monthly_swp, "Currency"))

ws_summary.cell(row=5, column=2, value="Strategic Variable KPI").font = header_font; ws_summary.cell(row=5, column=2).fill = header_fill
ws_summary.cell(row=5, column=3, value="Model Assessment Metrics").font = header_font; ws_summary.cell(row=5, column=3).fill = header_fill

cursor = 6
for kpi, val, val_type in summary_kpis:
    ck = ws_summary.cell(row=cursor, column=2, value=kpi); cv = ws_summary.cell(row=cursor, column=3, value=val)
    ck.font = regular_font; ck.border = thin_border; cv.border = thin_border
    if val_type == "Currency": cv.number_format = '₹#,##0.00'; cv.alignment = right_align
    elif val_type == "Percentage": cv.number_format = '0.00%'; cv.alignment = right_align
    elif val_type == "Integer": cv.number_format = '#,##0'; cv.alignment = center_align
    elif val_type == "Alert":
        cv.font = Font(name="Calibri", size=11, bold=True, color="9C0006"); cv.fill = PatternFill(fill_type="solid", start_color="FFC7CE")
        cv.alignment = center_align
    elif val_type == "StatusText":
        cv.font = Font(name="Calibri", size=11, bold=True, color="006100"); cv.fill = PatternFill(fill_type="solid", start_color="C6EFCE")
        cv.alignment = center_align
    else: cv.alignment = Alignment(horizontal="left")
    cursor += 1

# Render Strategic Advisor Note Block 
note_start = cursor + 1; note_end = note_start + 4
ws_summary.merge_cells(start_row=note_start, start_column=2, end_row=note_end, end_column=3)
note_cell = ws_summary.cell(row=note_start, column=2, value=f"💡 STRATEGIC ADVISORY INCOME CAPACITY NOTE:\n{summary['Client Presentation Note']}")
note_cell.font = Font(name="Calibri", size=10, italic=True, color="1F497D"); note_cell.alignment = Alignment(horizontal="left", vertical="top", wrap_text=True)
for r in range(note_start, note_end + 1):
    for c in [2, 3]: ws_summary.cell(row=r, column=c).border = thin_border

# Populate Sheet 2 (Annual Grid)
ws_projection.cell(row=2, column=1, value="40-Year Annual Audit Ledger Matrix").font = title_font
for col_num, h_text in enumerate(list(projection_matrix.columns), start=1):
    c = ws_projection.cell(row=4, column=col_num, value=h_text); c.font = header_font; c.fill = header_fill; c.alignment = center_align

for r_idx, row_data in enumerate(projection_matrix.itertuples(index=False), start=5):
    for c_idx, value in enumerate(row_data, start=1):
        cell = ws_projection.cell(row=r_idx, column=c_idx, value=value); cell.font = regular_font; cell.border = thin_border
        if r_idx % 2 == 0: cell.fill = PatternFill(start_color="F9FAFB", fill_type="solid")
        if c_idx in [1, 2]: cell.alignment = center_align; cell.number_format = '#,##0'
        elif c_idx == 11:
            cell.alignment = center_align
            cell.font = Font(name="Calibri", size=10, color="9C0006" if value == "Corpus Exhausted" else "006100")
            cell.fill = PatternFill(fill_type="solid", start_color="FFC7CE" if value == "Corpus Exhausted" else "C6EFCE")
        else: cell.alignment = right_align; cell.number_format = '₹#,##0.00'

# Populate Sheet 3 (MoM Bridge Grid - Gated)
if is_unsustainable and ws_mom is not None:
    ws_mom.cell(row=2, column=1, value="Month-on-Month Deficit Bridge Accumulation Ledger").font = title_font
    for col_num, h_text in enumerate(list(mom_matrix.columns), start=1):
        c = ws_mom.cell(row=4, column=col_num, value=h_text); c.font = header_font; c.fill = header_fill; c.alignment = center_align
    for r_idx, row_data in enumerate(mom_matrix.itertuples(index=False), start=5):
        for c_idx, value in enumerate(row_data, start=1):
            cell = ws_mom.cell(row=r_idx, column=c_idx, value=value); cell.font = regular_font; cell.border = thin_border
            if r_idx % 2 == 0: cell.fill = PatternFill(start_color="F9FAFB", fill_type="solid")
            if c_idx in [1, 2]: cell.alignment = center_align; cell.number_format = '#,##0'
            else:
                cell.alignment = right_align
                if isinstance(value, (int, float)):
                    cell.number_format = '₹#,##0.00'
                    if c_idx == 7: cell.font = Font(name="Calibri", size=11, bold=True, italic=True, color="1F497D")

# Populate Sheet 4 (Images)
ws_graphs.cell(row=2, column=2, value="VISUAL STRATEGIC ANCHORS").font = title_font
if os.path.exists("chart1.png"): ws_graphs.add_image(Image("chart1.png"), "B4")
if is_unsustainable and os.path.exists("chart2.png"): ws_graphs.add_image(Image("chart2.png"), "B28")

# Formatting Cleanups & Autosizing
for sheet in [ws_summary, ws_projection] + ([ws_mom] if ws_mom else []):
    for col in sheet.columns:
        col_letter = openpyxl.utils.get_column_letter(col[0].column)
        sheet.column_dimensions[col_letter].width = 18
ws_summary.column_dimensions['B'].width = 34; ws_summary.column_dimensions['C'].width = 26

wb.save(excel_filename)
print(f"Success! Generated spreadsheet workbook package: '{excel_filename}'")

Success! Generated spreadsheet workbook package: 'Rahul_Sharma_Wealth_Plan.xlsx'


In [14]:
# ═════════════════════════════════════════════════════════════════════════════════════════
# 📑 PART 4: ONE-CLICK PDF GENERATION MODULE (WITH BRANDING & LOGO)
# ═════════════════════════════════════════════════════════════════════════════════════════
import os
import urllib.request
from io import BytesIO
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.pdfgen import canvas

class WatermarkCanvas(canvas.Canvas):
    """
    Custom canvas layout to stamp a pristine, angled corporate watermark
    transparently into the background layer of every generated page.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pages = []

    def showPage(self):
        self.pages.append(dict(self.__dict__))
        self._startPage()

    def save(self):
        for page in self.pages:
            self.__dict__.update(page)
            self.draw_watermark()
            super().showPage()
        super().save()

    def draw_watermark(self):
        self.saveState()
        self.setFont("Helvetica-Bold", 38)
        self.setFillColor(colors.HexColor("#E5E7EB")) 
        self.setFillAlpha(0.12)  # Transparent enough to sit behind numbers cleanly
        
        center_x = 4.25 * inch
        center_y = 5.5 * inch
        
        self.translate(center_x, center_y)
        self.rotate(45)
        self.drawCentredString(0, 0, "Livlong Insurance Brokers Limited")
        self.restoreState()


def generate_client_pdf(inputs: dict, summary_data: dict, chart_path: str = "chart1.png"):
    """
    Generates an executive-ready 2-page presentation report featuring an integrated
    corporate logo letterhead header, custom table datasets, and background watermarks.
    """
    pdf_filename = f"{inputs['client_name'].replace(' ', '_')}_Execution_Summary.pdf"
    
    # Setup Document Template (0.5 inch margins)
    doc = SimpleDocTemplate(
        pdf_filename,
        pagesize=letter,
        leftMargin=36, rightMargin=36,
        topMargin=36, bottomMargin=36
    )
    
    story = []
    
    # Corporate Color Palette
    PRIMARY_COLOR = colors.HexColor("#1F497D")    # Deep Corporate Blue
    TEXT_COLOR = colors.HexColor("#222222")       # Charcoal Body Text
    ACCENT_RED = colors.HexColor("#9C0006")       # Alert Red
    ACCENT_GREEN = colors.HexColor("#006100")     # Sustainable Green
    BG_LIGHT = colors.HexColor("#F9FAFB")         # Soft Neutral Background
    
    base_styles = getSampleStyleSheet()
    
    # Typography Configurations
    title_style = ParagraphStyle(
        'DocTitle', parent=base_styles['Normal'],
        fontName='Helvetica-Bold', fontSize=22, textColor=PRIMARY_COLOR,
        leading=26, spaceAfter=4
    )
    subtitle_style = ParagraphStyle(
        'DocSub', parent=base_styles['Normal'],
        fontName='Helvetica', fontSize=11, textColor=colors.HexColor("#444444"),
        leading=14, spaceAfter=20
    )
    company_header_style = ParagraphStyle(
        'CompHeader', parent=base_styles['Normal'],
        fontName='Helvetica-Bold', fontSize=12, textColor=PRIMARY_COLOR,
        alignment=2, leading=14  # Right-aligned
    )
    section_heading = ParagraphStyle(
        'SectionHead', parent=base_styles['Normal'],
        fontName='Helvetica-Bold', fontSize=14, textColor=PRIMARY_COLOR,
        spaceBefore=14, spaceAfter=10
    )
    body_style = ParagraphStyle(
        'BodyDark', parent=base_styles['Normal'],
        fontName='Helvetica', fontSize=10, textColor=TEXT_COLOR, leading=14
    )
    table_header_style = ParagraphStyle(
        'THead', parent=base_styles['Normal'],
        fontName='Helvetica-Bold', fontSize=10, textColor=colors.white, alignment=1
    )
    advice_note_style = ParagraphStyle(
        'AdviceNote', parent=base_styles['Normal'],
        fontName='Helvetica-Oblique', fontSize=10.5, textColor=PRIMARY_COLOR, leading=16
    )

    # ═════════════════════════════════════════════════════════════════════════
    # 🌟 NEW FEATURE: LIVE LETTERHEAD GENERATION WITH LOGO
    # ═════════════════════════════════════════════════════════════════════════
    try:
        logo_url = "https://assets.livlong.com/static-images/gmc/LL-INSURANCE-LOGO1.png"
        
        # Request headers to bypass restrictive asset firewall blocks
        req = urllib.request.Request(logo_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            logo_data = BytesIO(response.read())
        
        # Pull and downscale the image safely to fit a clean header bar layout
        logo_img = RLImage(logo_data, width=1.5 * inch, height=0.45 * inch)
    except Exception as e:
        # Fallback element if your computer is completely disconnected from the internet
        logo_img = Paragraph("<b>[LIVLONG LOGO]</b>", body_style)

    company_name_p = Paragraph("<b>Livlong Insurance Brokers Limited</b><br/><font size=8 color='#555555'>Wealth Advisory Division</font>", company_header_style)
    
    # Layout header table to keep logo left and company name right
    header_table = Table([[logo_img, company_name_p]], colWidths=[3.75 * inch, 3.75 * inch])
    header_table.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('ALIGN', (0,0), (0,0), 'LEFT'),
        ('ALIGN', (1,0), (1,0), 'RIGHT'),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LINEBELOW', (0,0), (-1,-1), 1, colors.HexColor("#E5E7EB")), # Soft dividing line
    ]))
    story.append(header_table)
    story.append(Spacer(1, 15))

    # ═════════════════════════════════════════════════════════════════════════
    # PAGE 1 CONTENT: STRATEGIC BRIEF
    # ═════════════════════════════════════════════════════════════════════════
    story.append(Paragraph("STRATEGIC WEALTH ARCHITECTURE PLAN", title_style))
    story.append(Paragraph(f"<b>Client Profile:</b> {inputs['client_name']}  |  <b>Confidential Portfolio Forecast</b>", subtitle_style))
    story.append(Spacer(1, 5))
    
    story.append(Paragraph("Key Model Assessment Metrics", section_heading))
    
    is_red = summary_data["Sustainability Status"] == "Corpus Exhausted"
    status_color = ACCENT_RED if is_red else ACCENT_GREEN
    
    # Table Content
    kpi_rows = [
        [Paragraph("<b>Financial Vector Variable</b>", table_header_style), Paragraph("<b>Target Valuation Metrics</b>", table_header_style)],
        [Paragraph("Current Age", body_style), Paragraph(f"{inputs['current_age']} Years", body_style)],
        [Paragraph("Expected Portfolio Returns", body_style), Paragraph(f"{inputs['expected_return']*100:.1f}% Annualized", body_style)],
        [Paragraph("Target Retirement Milestone Age", body_style), Paragraph(f"{inputs['swp_start_age']} Years Old", body_style)],
        [Paragraph("Initial Monthly Insurance Payout Stream", body_style), Paragraph(f"Rs. {summary_data['Monthly Insurance Payout']:,.2f}", body_style)],
        [Paragraph("Total Guaranteed Life Cover Amount", body_style), Paragraph(f"Rs. {summary_data['Life Cover Amount']:,.2f}", body_style)],
        [Paragraph(f"Accumulated Capital at Age {inputs['swp_start_age']}", body_style), Paragraph(f"Rs. {summary_data['Corpus at SWP Start']:,.2f}", body_style)],
        [Paragraph("Required Corpus Target for Target SWP", body_style), Paragraph(f"Rs. {summary_data['Target Required Corpus']:,.2f}", body_style)],
        [Paragraph("Calculated Surplus / Shortfall Gap", body_style), Paragraph(f"Rs. {summary_data['Surplus / Shortfall']:,.2f}", body_style)],
        [Paragraph("Strategy Sustainability Status", body_style), Paragraph(f"<b>{summary_data['Sustainability Status'].upper()}</b>", ParagraphStyle('Status', parent=body_style, textColor=status_color, fontName='Helvetica-Bold'))],
    ]
    
    if is_red:
        from __main__ import bridge_calculator
        kpi_rows.append([Paragraph("<b>Required Monthly Wealth Accelerator Top-Up</b>", body_style), Paragraph(f"<b>Rs. {bridge_calculator.required_monthly_investment:,.2f}/mo</b>", ParagraphStyle('AlertText', parent=body_style, fontName='Helvetica-Bold', textColor=ACCENT_RED))])
    else:
        from __main__ import max_sustainable_monthly_swp
        kpi_rows.append([Paragraph("<b>Maximum Safe Monthly SWP Capacity</b>", body_style), Paragraph(f"<b>Rs. {max_sustainable_monthly_swp:,.2f}/mo</b>", ParagraphStyle('SafeText', parent=body_style, fontName='Helvetica-Bold', textColor=ACCENT_GREEN))])

    # Table Compilation
    metrics_table = Table(kpi_rows, colWidths=[4.0 * inch, 3.5 * inch])
    metrics_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (1, 0), PRIMARY_COLOR),
        ('ALIGN', (0, 0), (-1, -1), 'LEFT'),
        ('VALIGN', (0, 0), (-1, -1), 'MIDDLE'),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
        ('TOPPADDING', (0, 0), (-1, -1), 6),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, BG_LIGHT]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor("#E5E7EB")),
        ('LINEBELOW', (0, -1), (-1, -1), 1.5, PRIMARY_COLOR),
    ]))
    story.append(metrics_table)
    story.append(Spacer(1, 15))
    
    story.append(Paragraph("Strategic Advice", section_heading))
    sanitized_note = summary_data['Client Presentation Note'].replace("₹", "Rs. ")
    
    note_p = Paragraph(f"💡 <b>Advisor Analysis Note:</b><br/>{sanitized_note}", advice_note_style)
    note_box_table = Table([[note_p]], colWidths=[7.5 * inch])
    note_box_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (0, 0), colors.HexColor("#EFF6FF")), 
        ('BOX', (0, 0), (0, 0), 1, colors.HexColor("#BFDBFE")),
        ('TOPPADDING', (0, 0), (0, 0), 12),
        ('BOTTOMPADDING', (0, 0), (0, 0), 12),
        ('LEFTPADDING', (0, 0), (0, 0), 14),
        ('RIGHTPADDING', (0, 0), (0, 0), 14),
    ]))
    story.append(note_box_table)
    
    story.append(PageBreak())
    
    # ═════════════════════════════════════════════════════════════════════════
    # PAGE 2 CONTENT: GRAPH MAPPING
    # ═════════════════════════════════════════════════════════════════════════
    # We re-inject the branding header on page 2 to maintain institutional design consistency
    story.append(header_table)
    story.append(Spacer(1, 15))
    
    story.append(Paragraph("Long-Term Wealth Projection Mapping", section_heading))
    story.append(Paragraph("The chart below illustrates the visual timeline trajectory of your financial portfolio asset engine over the specified 40-year duration scope.", body_style))
    story.append(Spacer(1, 15))
    
    if os.path.exists(chart_path):
        chart_img = RLImage(chart_path, width=7.5 * inch, height=3.75 * inch)
        story.append(chart_img)
    else:
        story.append(Paragraph("[Visual Chart Graphic Asset Not Loaded]", body_style))
        
    story.append(Spacer(1, 25))
    
    footer_text = Paragraph(
        "<font color='#777777'><b>Disclaimer:</b> This projection summary document models mathematical simulations "
        "based on asset run rates and step-up logic parameters chosen. Actual market return behaviors may introduce variables. "
        "Please consult your designated executive planner for rolling adjustments.</font>", 
        ParagraphStyle('Footer', parent=body_style, fontSize=8, textColor=colors.HexColor("#777777"), alignment=4)
    )
    story.append(footer_text)
    
    # Compile Everything with Watermark Template
    doc.build(story, canvasmaker=WatermarkCanvas)
    print(f"Success! Perfect institutional handout created: '{pdf_filename}'")

# Re-run generation layout
generate_client_pdf(advisor_inputs, summary, "chart1.png")

Success! Perfect institutional handout created: 'Rahul_Sharma_Execution_Summary.pdf'
